# Chapter 2 — Electromagnetics & RF Propagation — equations

Standalone, runnable subset of the master `../RF_Equations.ipynb`, scoped to this chapter.
Run top-to-bottom: **Setup**, then this chapter's sections. All functions are verified against the book's worked examples.

## Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

C = 2.99792458e8        # speed of light, m/s
EPS0 = 8.8541878128e-12 # vacuum permittivity, F/m

def wavelength(f_hz):
    return C / f_hz


## 4. Material Properties: Permittivity, Conductivity, Index — Ch 2.2-2.3

The engine's whole material model reduces to two numbers per material — relative
permittivity `eps_r` and conductivity `sigma` — because the book assumes **mu_r = 1**
(non-magnetic, §2.3). That single assumption is why the refractive index is
`n = sqrt(eps_r)` and why the Fresnel coefficients (§6) take `eps_r` alone.

- `eps_r` (dielectric constant, Table 2.2) -> the *real* part; sets reflection / refraction.
- `sigma` S/m (Table 2.3) -> feeds the *imaginary* part (complex permittivity, §5); sets loss.
- Constitutive: `E = D/eps`, `B = mu*H`. Boundary rules: normal `D` continuous, tangential `E` continuous.


In [ ]:
MU0 = 4*np.pi*1e-7   # H/m, vacuum permeability (EPS0 is defined in Setup)

# Relative permittivity (Table 2.2) and conductivity S/m (Table 2.3). Source: Seybold/Plonus.
# A complete engine entry needs BOTH; these are the materials the book pairs up.
MATERIALS = {   # name: (eps_r, sigma_S_per_m)
    "vacuum":          (1.0,    0.0),
    "air":             (1.0006, 0.0),
    "rubber":          (3.0,    1e-15),
    "quartz":          (5.0,    1e-17),
    "lead_glass":      (6.0,    1e-12),   # eps_r: lead glass; sigma: generic glass ~1e-12
    "mica":            (6.0,    1e-15),
    "distilled_water": (81.0,   1e-4),
}
# eps_r only (book gives no sigma): polystyrene 2.7, bakelite 5
# sigma only (conductors/earth): silver 6.1e7, copper 5.7e7, aluminum 3.5e7, seawater 4,
#   wet_earth ~1e-3, dry_earth ~1e-5, rock ~1e-6   (metals ~1e7 => near-perfect reflectors)

def refractive_index(eps_r, mu_r=1.0):
    # n = sqrt(mu_r * eps_r). With the book's mu_r = 1, n = sqrt(eps_r).
    return np.sqrt(eps_r*mu_r)

for name, (er, sig) in MATERIALS.items():
    print(f"{name:16s} eps_r={er:<7g} sigma={sig:<9g} n={refractive_index(er):.3f}")


In [ ]:
# Static E-FIELD refraction at a dielectric boundary (Ch 2.2, eqs 2.1-2.2).
# WARNING: this bends the E-FIELD VECTOR, not the propagation ray. It is NOT Snell's law
# (the book footnotes the distinction). It even bends the OPPOSITE sense to a ray: the E
# field tilts AWAY from the normal into higher eps_r, whereas a ray bends TOWARD the normal.
# Wave/ray refraction (Snell) comes from the boundary physics of Ch 2.6 -> see Fresnel (§6).
def efield_refraction_angle(phi1_rad, eps_r2, eps_r1=1.0):
    # Angle of E from the boundary normal in medium 2, given the angle in medium 1.
    return np.arctan((eps_r2/eps_r1)*np.tan(phi1_rad))

phi1 = np.radians(np.linspace(0, 89, 200))
plt.figure(figsize=(6,4))
for er2 in (3, 6, 81):
    plt.plot(np.degrees(phi1), np.degrees(efield_refraction_angle(phi1, er2)), label=f"eps_r2={er2}")
plt.plot(np.degrees(phi1), np.degrees(phi1), "k--", lw=0.8, label="no bend")
plt.xlabel("E angle from normal, medium 1 (deg)")
plt.ylabel("E angle from normal, medium 2 (deg)")
plt.title("Static E-field refraction (eq 2.1) — NOT Snell"); plt.legend(); plt.grid(True); plt.show()

print("phi1=0 ->", np.degrees(efield_refraction_angle(0.0, 81)), "deg (unrefracted)")
print("phi1=45deg, eps_r2=81 ->", round(float(np.degrees(efield_refraction_angle(np.radians(45), 81))),2), "deg")


## 5. Waves in Matter: Complex ε, Attenuation, Skin Depth — Ch 2.4

$$\varepsilon_c = \varepsilon' - j\varepsilon'' = \varepsilon_r - j\frac{\sigma}{\omega\varepsilon_0},
\qquad \tan\delta = \frac{\sigma}{\omega\varepsilon_r\varepsilon_0}$$

**Engine hook:** the imaginary part is *absorption*. Feed `eps_c` straight into the
Fresnel coefficients (next) so lossy walls both reflect and attenuate.


In [ ]:
def complex_permittivity(eps_r, sigma, f_hz):
    # Relative complex permittivity eps_c (dimensionless).
    w = 2*np.pi*f_hz
    return eps_r - 1j*sigma/(w*EPS0)

def loss_tangent(eps_r, sigma, f_hz):
    return sigma/(2*np.pi*f_hz*eps_r*EPS0)

# concrete-ish wall @ 2.4 GHz: eps_r ~ 5.24, sigma ~ 0.15 S/m
print("eps_c =", complex_permittivity(5.24, 0.15, 2.4e9))
print("tan(delta) =", loss_tangent(5.24, 0.15, 2.4e9))


**§2.4 wave-in-matter quantities** — all from the same material constants:
phase velocity `v = c/√(εr·μr)` (eq 2.4) · intrinsic impedance `Z0 = 377·√(μr/εr) Ω` (eq 2.12) ·
attenuation `α` + phase `β` from the complex wave number (eqs 2.9a/2.9b) · good-conductor
skin depth `δ = 1/√(π f μ σ)` (eq 2.11). Loss regime by tanδ: `<0.1` low (dielectric-like),
`>10` high (conductor-like).


In [ ]:
def phase_velocity(eps_r, mu_r=1.0):
    return C/np.sqrt(eps_r*mu_r)                 # eq 2.4

def intrinsic_impedance(eps_r, mu_r=1.0):
    return 377.0*np.sqrt(mu_r/eps_r)             # eq 2.12

def loss_regime(eps_r, sigma, f_hz):
    lt = loss_tangent(eps_r, sigma, f_hz)
    tag = ("low-loss (dielectric)" if lt < 0.1 else
           "high-loss (conductor)" if lt > 10 else "intermediate")
    return tag, lt

for name, er, sig in [("lead_glass",6,1e-12), ("distilled_water",81,1e-4), ("aluminum~",1.0,3.5e7)]:
    tag, lt = loss_regime(er, sig, 2.4e9)
    print(f"{name:15s} v={phase_velocity(er)/1e8:.2f}e8 m/s  Z0={intrinsic_impedance(er):6.1f} ohm  tanδ={lt:.1e} -> {tag}")


In [ ]:
def wave_params_lossy(eps_r, sigma, f_hz, mu_r=1.0):
    # Attenuation alpha (Np/m) and phase beta (rad/m) in a lossy medium. Eqs 2.9a/2.9b.
    w = 2*np.pi*f_hz
    eps = eps_r*EPS0; mu = mu_r*MU0
    root = np.sqrt(1 + (sigma/(w*eps))**2)
    alpha = w*np.sqrt(mu*eps/2*(root - 1))
    beta  = w*np.sqrt(mu*eps/2*(root + 1))
    return alpha, beta

def skin_depth(sigma, f_hz, mu_r=1.0):
    # Good-conductor skin depth (m): delta = 1/sqrt(pi f mu sigma). Eq 2.11.
    return 1.0/np.sqrt(np.pi*f_hz*mu_r*MU0*sigma)

for f in (1e6, 2.4e9, 60e9):
    print(f"copper skin depth @ {f/1e9:8.4f} GHz = {skin_depth(5.7e7, f)*1e6:7.3f} um")
a, b = wave_params_lossy(5.24, 0.05, 2.4e9)
print(f"lossy wall (eps_r=5.24, sigma=0.05) @2.4GHz: alpha={a:.2f} Np/m = {8.686*a:.1f} dB/m")


## 6. Fresnel Reflection & Transmission — Ch 2.6

From medium 1 (default air) into medium 2 with relative permittivity `eps_r2`
(may be complex, from §5). With $n=\sqrt{\varepsilon_r}$ and
$\cos\theta_t=\sqrt{1-(n_1/n_2)^2\sin^2\theta_i}$:

$$\Gamma_{TE}=\frac{n_1\cos\theta_i-n_2\cos\theta_t}{n_1\cos\theta_i+n_2\cos\theta_t},
\qquad
\Gamma_{TM}=\frac{n_2\cos\theta_i-n_1\cos\theta_t}{n_2\cos\theta_i+n_1\cos\theta_t}$$

Power transmitted (lossless) $T = 1-|\Gamma|^2$. Brewster angle: $\Gamma_{TM}=0$.

**Engine hook:** `Gamma` is the per-surface reflection coefficient for the
reflection/refraction effects; `T` is the pass-through attenuation for absorption.


In [ ]:
def fresnel_coeffs(theta_i_rad, eps_r2, eps_r1=1.0):
    # Returns (Gamma_TE, Gamma_TM) reflection coefficients (complex).
    n1 = np.sqrt(np.asarray(eps_r1, complex))
    n2 = np.sqrt(np.asarray(eps_r2, complex))
    cos_i = np.cos(theta_i_rad)
    sin_t = (n1/n2)*np.sin(theta_i_rad)
    cos_t = np.sqrt(1 - sin_t**2)
    g_te = (n1*cos_i - n2*cos_t)/(n1*cos_i + n2*cos_t)
    g_tm = (n2*cos_i - n1*cos_t)/(n2*cos_i + n1*cos_t)
    return g_te, g_tm

theta = np.radians(np.linspace(0, 89.9, 400))
g_te, g_tm = fresnel_coeffs(theta, eps_r2=5.24)   # lossless concrete-ish
brewster = np.degrees(np.arctan(np.sqrt(5.24)))

plt.figure(figsize=(6,4))
plt.plot(np.degrees(theta), np.abs(g_te), label="|Γ_TE| (perp)")
plt.plot(np.degrees(theta), np.abs(g_tm), label="|Γ_TM| (parallel)")
plt.axvline(brewster, ls="--", c="gray", label=f"Brewster ≈ {brewster:.1f}°")
plt.xlabel("incidence angle (deg)"); plt.ylabel("|Γ|")
plt.title("Fresnel reflection, εr=5.24"); plt.legend(); plt.grid(True); plt.show()

# lossy wall: reflection stays high, transmitted power carries attenuation
eps_c = complex_permittivity(5.24, 0.15, 2.4e9)
g_te_l, _ = fresnel_coeffs(np.radians(30), eps_c)
print("|Γ_TE|@30° lossy =", abs(g_te_l), "  T ≈", 1 - abs(g_te_l)**2)


### §2.6 reconciliation — grazing vs normal, and the Brewster cross-check

The book derives Γ via a **transmission-line analogy** (effective wave impedances Z_L, Z_z1)
using the **grazing angle** (measured from the surface). `fresnel_coeffs()` above is the
equivalent standard-optics form measured **from the normal** (grazing = 90° − normal), so the
book's Fig 2.7 x-axis is the mirror of ours. Same physics — checks below:
- the book's critical/polarizing angle (eq 2.23, grazing) and our Brewster (arctan√εr, from
  normal) are **complementary** (sum to 90°);
- at **grazing** incidence Γ → −1 for **both** polarizations (the two-ray ground null);
- perfect-conductor limit |Γ| → 1.

Field transmission `τ = 1+Γ` (can exceed 1) vs power transmittance `T = 1−|Γ|²` (≤1) are
different quantities — don't mix them.


In [ ]:
def brewster_grazing_deg(eps_r2, eps_r1=1.0):
    return np.degrees(np.arcsin(np.sqrt(eps_r1/(eps_r1+eps_r2))))   # book eq 2.23 (from surface)

def brewster_normal_deg(eps_r2, eps_r1=1.0):
    return np.degrees(np.arctan(np.sqrt(eps_r2/eps_r1)))            # optics (from normal)

er = 5.24
g, nrm = brewster_grazing_deg(er), brewster_normal_deg(er)
print(f"Brewster: grazing(eq 2.23)={g:.1f} deg + normal(arctan√εr)={nrm:.1f} deg = {g+nrm:.1f}")

gte, gtm = fresnel_coeffs(np.radians(89.9), er)          # near grazing
print(f"near-grazing Γ: TE={gte.real:.3f}, TM={gtm.real:.3f}  (both -> -1)")

gte_c, gtm_c = fresnel_coeffs(np.radians(45), complex_permittivity(1.0, 1e7, 2.4e9))
print(f"conductor |Γ| @45deg: TE={abs(gte_c):.3f}, TM={abs(gtm_c):.3f}")
